# 01. Agentic RL과 sparse reward 기초

목표: SEED가 해결하려는 supervision gap을 작은 trajectory 예제로 이해한다.

실행 방법:
1. Jupyter Notebook 또는 VS Code에서 이 파일을 연다.
2. 위에서 아래로 셀을 실행한다.
3. 이 노트북은 Python 표준 라이브러리만 사용한다.

이 실습은 공식 SEED 구현이 아니다. 논문의 핵심 개념을 손으로 계산해 보는 교육용 축소 예제다.

In [ ]:
from dataclasses import dataclass
from statistics import mean, pstdev


@dataclass
class Step:
    observation: str
    action: str
    reward: float = 0.0


@dataclass
class Trajectory:
    task: str
    steps: list[Step]
    outcome: float


def show_trajectory(traj):
    print("task:", traj.task)
    for index, step in enumerate(traj.steps, start=1):
        print(f"  t={index}: obs={step.observation!r}, action={step.action!r}, reward={step.reward}")
    print("final outcome:", traj.outcome)

## 1. 완료된 trajectory 만들기

장기 에이전트 task에서는 중간 reward가 모두 0이고, 마지막에만 성공 여부가 주어지는 경우가 많다. 이것이 sparse trajectory-level reward다.

In [ ]:
trajectories = [
    Trajectory(
        task="find evidence and answer",
        steps=[
            Step("question mentions two entities", "search broad query"),
            Step("results are noisy", "refine query with entity names"),
            Step("source contains answer", "answer with citation"),
        ],
        outcome=1.0,
    ),
    Trajectory(
        task="find evidence and answer",
        steps=[
            Step("question mentions two entities", "guess from memory"),
            Step("environment asks for evidence", "repeat answer"),
            Step("answer unsupported", "final answer"),
        ],
        outcome=0.0,
    ),
]

for traj in trajectories:
    show_trajectory(traj)
    print()

## 2. Supervision gap 확인

최종 outcome만 보면 성공 trajectory의 모든 action이 좋은 것처럼 보이고, 실패 trajectory의 모든 action이 나쁜 것처럼 보인다. 하지만 실제로는 성공 trajectory에도 불필요한 action이 있을 수 있고, 실패 trajectory에도 유용한 partial behavior가 있을 수 있다.

In [ ]:
def broadcast_outcome_to_steps(traj):
    """trajectory-level outcome을 모든 step에 그대로 붙인다.

    이 방식은 단순하지만, 어떤 중간 action이 실제로 좋았는지 구분하지 못한다.
    SEED는 이 간극을 hindsight skill 기반 token-level signal로 보완하려 한다.
    """
    return [(step.action, traj.outcome) for step in traj.steps]


for traj in trajectories:
    print(traj.task, "outcome labels:")
    for action, label in broadcast_outcome_to_steps(traj):
        print(f"  {action:32s} -> {label}")
    print()

## 3. Group-relative advantage

SEED는 RL term으로 GRPO 계열 objective를 사용한다. 핵심 직관은 같은 task에서 여러 rollout을 만들고, 그 group 안에서 outcome이 평균보다 좋은지 나쁜지를 advantage로 쓰는 것이다.

In [ ]:
group_outcomes = [1.0, 0.0, 1.0, 0.0, 0.5, 1.0, 0.0, 0.5]


def group_relative_advantages(outcomes, epsilon=1e-8):
    """그룹 평균과 표준편차로 정규화한 advantage를 계산한다."""
    mu = mean(outcomes)
    sigma = pstdev(outcomes)
    return [(outcome - mu) / (sigma + epsilon) for outcome in outcomes]


advantages = group_relative_advantages(group_outcomes)
for outcome, advantage in zip(group_outcomes, advantages):
    print(f"outcome={outcome:.1f}, advantage={advantage:+.3f}")

## 4. Hindsight skill의 역할

완료된 trajectory를 보면 성공과 실패의 이유를 자연어 skill로 요약할 수 있다. SEED는 이 skill을 추론 때 붙이지 않고, 학습 중 token-level supervision으로 바꿔 policy에 내재화한다.

In [ ]:
def hindsight_skill(traj):
    """완료된 trajectory에서 학습용 skill을 추출하는 장난감 analyzer다."""
    actions = " ".join(step.action for step in traj.steps)
    if traj.outcome > 0 and "refine query" in actions:
        return "Start broad, refine the query using entities, then answer with cited evidence."
    if traj.outcome == 0 and "guess" in actions:
        return "Avoid guessing from memory; gather evidence before committing to an answer."
    return "Use observations to choose the next information-gathering action."


for traj in trajectories:
    print("outcome:", traj.outcome)
    print("skill:", hindsight_skill(traj))
    print()

## 정리

- long-horizon agent task에서는 final outcome만으로 중간 action을 세밀하게 지도하기 어렵다.
- group-relative advantage는 같은 task의 여러 rollout을 상대 비교한다.
- hindsight skill은 completed trajectory에서 사후적으로 얻은 자연어 교훈이다.
- 다음 노트북에서는 skill이 token log-probability를 어떻게 dense OPD signal로 바꾸는지 계산한다.